Soil grain size from photos — pipeline v5

 STARTING POINT: your teammate's notebook (the one that scored 40.38 -- your best result).
 It uses classical, physically-calibrated texture features (LBP, GLCM, multi-scale edge
 density, Otsu-threshold blob segmentation, FFT dominant scale, Laplacian/gradient
 statistics) plus simple, heavily regularised regressors (ridge, elastic net, PLS,
 Bayesian ridge) -- NOT deep embeddings. That is a good instinct: my own diagnosis (in the
 earlier v3/v4 notebooks) found that DINOv2/CLIP embeddings respond more to which iPhone
 took a test photo than to the soil itself, so a model built on them is partly a camera
 classifier. Classical, ppm-calibrated texture descriptors on a CLAHE-contrast-normalised
 image are much less likely to carry that fingerprint.

 A BUG FOUND IN THE TEAMMATE'S SCRIPT, CONFIRMED AGAINST THE REAL DIRECTORY LAYOUT:
   TEST_IMG_DIR = f"{DATA_DIR}/Test_All_Photos/Test_All_Photos"   (double-nested)
 The real layout has NO double nesting (only the TRAINING folder is double-nested; verified
 directly against the actual competition file listing). So that script's test-photo glob
 finds ZERO files. Tracing the fallback path: when the test feature table ends up empty,
 every one of the 10 submission rows is silently filled with the SAME constant -- the
 monotonic-enforced TRAINING-MEAN curve. In other words, the 40.38 score is almost
 certainly "always predict the average training curve", not a content-based prediction at
 all. That is a genuinely useful, if humbling, data point: it means every content-based
 model tried so far (including my own DINOv2/CLIP-based ones, which scored 68-73) did
 WORSE than doing nothing. This raises the real bar: a model has to beat the plain mean
 curve to be worth submitting, and that is checked explicitly and honestly below, rather
 than assumed.

 THE PLAN FOR v5:
   1. Fix the path bug: list photos with a robust recursive glob (no hardcoded nesting
      assumption) for both train and test.
   2. Reuse the teammate's full, proven feature-extraction code (LBP/GLCM/edges/blobs/FFT/
      Laplacian/gradients) essentially verbatim.
   3. Add a second, complementary physical feature block: a radial power spectrum (grain
      scale as a frequency-domain fingerprint) and an independent Canny-based blob
      segmentation (a second, differently-biased estimate of visible particle size,
      alongside the teammate's Otsu-based one). More than one measurement of the same
      physical quantity, taken differently, tends to combine well when neither is perfect.
   4. Keep DINOv2 as an optional, separately-graded feature set (matching the teammate's
      structure) -- it is allowed to compete, but is not assumed to help.
   5. Broaden the candidate model zoo: the teammate's 10 regressors, PLUS a k-nearest-
      neighbour-over-samples model (won clearly in my own earlier feature space) and kernel
      ridge, PLUS the plain mean/median curve as EXPLICIT candidates -- so the honest CV can
      show, in black and white, whether anything actually beats guessing the average.
   6. Full nested cross-validation: unlike the teammate's script (which nests the
      model+transform choice only, on top of a feature set picked without nesting), this
      re-selects the feature set AND the model AND the transform inside every outer fold.
      That is a stronger, more honest generalisation estimate given only 24 training
      samples, where naive selection can look better than it really is.
   7. A leave-one-camera-out check (Motorola vs Samsung) as a second, independent signal --
      the closest available proxy for a real camera-domain shift, since LOOCV alone cannot
      see it (every training camera is present in every LOOCV fold).
   8. An optional blend of the top few distinct configurations, kept ONLY if it improves the
      nested score over the single best model -- ensembling is not assumed to help either.

 Needs internet for the DINOv2 download (optional -- set USE_DINO=False to skip it and run
 on the classical features alone, which is both faster and, per the diagnosis above, likely
 more trustworthy).

## 1. Setup, metric, transforms (mostly the teammate's proven code, reused as-is)

In [1]:
import os, re, glob, warnings, time
import numpy as np
import pandas as pd
import cv2
from skimage.feature import local_binary_pattern, graycomatrix, graycoprops
from sklearn.linear_model import Ridge, ElasticNet, MultiTaskElasticNet, MultiTaskLasso, BayesianRidge
from sklearn.cross_decomposition import PLSRegression
from sklearn.kernel_ridge import KernelRidge
from sklearn.model_selection import LeaveOneOut
from sklearn.preprocessing import StandardScaler
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics.pairwise import euclidean_distances
from collections import Counter
warnings.filterwarnings("ignore")

USE_DINO = True     # set False to skip the (diagnosed-as-risky) DINOv2 features and internet download

DATA_DIR = "/kaggle/input/competitions/soil-grain-size-from-photos"
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

SUPPORT_MM = np.array([0.002, 0.0063, 0.02, 0.063, 0.2, 0.63, 2.0, 6.3, 20.0, 63.0, 200.0])
SUPPORT_COLS = ["0.002", "0.0063", "0.02", "0.063", "0.2", "0.63", "2", "6.3", "20", "63", "200"]
LOG_X = np.log10(SUPPORT_MM)
LOG_WEIGHTS = np.diff(LOG_X)     # EXACT competition metric weights (left-endpoint rule)

def emd_log_score(y_true, y_pred):
    y_true, y_pred = np.atleast_2d(y_true), np.atleast_2d(y_pred)
    abs_diff = np.abs(y_true[:, :-1] - y_pred[:, :-1])
    return (abs_diff * LOG_WEIGHTS).sum(axis=1).mean()

def enforce_monotonic_and_bounds(pred_row):
    p = np.clip(np.asarray(pred_row, dtype=float), 0, 100)
    p = np.maximum.accumulate(p); p[-1] = 100.0; p = np.minimum(p, 100.0)
    return np.maximum.accumulate(p)

class CumulativeTransform:
    name = 'cumulative'
    def encode(self, Y): return Y.copy()
    def decode(self, Yt): return np.array([enforce_monotonic_and_bounds(r) for r in Yt])

class DeltaTransform:
    name = 'delta'
    def encode(self, Y): return np.diff(Y, axis=1, prepend=0)
    def decode(self, Dt):
        D = np.clip(Dt, 0, None)
        s = D.sum(axis=1, keepdims=True); s = np.where(s <= 1e-9, 1.0, s)
        F = np.cumsum(D / s * 100.0, axis=1)
        return np.array([enforce_monotonic_and_bounds(r) for r in F])

TRANSFORMS = [CumulativeTransform(), DeltaTransform()]

## 2. File listing — FIXED to use a robust recursive glob (no hardcoded nesting assumption)

In [2]:
def normalize_sample_id(s):
    s = s.replace('Ü','Ue').replace('ü','ue').replace('Ä','Ae').replace('ä','ae')
    s = s.replace('Ö','Oe').replace('ö','oe').replace('ß','ss').replace(',', '_')
    return s.strip()

def parse_train_filename(path):
    base = os.path.basename(path)
    m = re.search(r'([A-Za-z]\d{3})_(\d+)\.(jpg|jpeg)$', base, flags=re.IGNORECASE)
    if not m: return None, None
    return base[:m.start()].rstrip('_'), m.group(1)

def parse_test_filename(path):
    base = os.path.basename(path)
    n = re.sub(r'\.(jpg|jpeg)$', '', base, flags=re.IGNORECASE)
    n = re.sub(r'\s*\(\d+\)\s*$', '', n)
    idx = n.find('HPC')
    if idx == -1: return 'unknown', normalize_sample_id(n)
    return n[:idx].rstrip('_').rstrip(), normalize_sample_id(n[idx:])

def list_photos_recursive(root_dir, is_train):
    """FIX: search recursively under root_dir, at any depth, instead of assuming a fixed
    nesting level. This is what actually finds the 35 test photos (the teammate's script
    found 0 because of the hardcoded double-nested path -- verified against the real layout)."""
    paths = sorted(set(
        glob.glob(f"{root_dir}/**/*.jpg", recursive=True) + glob.glob(f"{root_dir}/**/*.JPG", recursive=True) +
        glob.glob(f"{root_dir}/**/*.jpeg", recursive=True) + glob.glob(f"{root_dir}/**/*.JPEG", recursive=True)
    ))
    out = []
    for path in paths:
        if is_train:
            camera, sample_id = parse_train_filename(path)
            if sample_id is None:
                print(f"WARNING: could not parse TRAIN filename {os.path.basename(path)}, skipping"); continue
        else:
            camera, sample_id = parse_test_filename(path)
        out.append((path, sample_id, camera))
    return out

## 3. ppm lookup (teammate's proven code, reused as-is)

In [3]:
def _canon(s): return re.sub(r'[^a-z0-9]', '', str(s).lower())

def load_ppm_table():
    df = pd.read_csv(f"{DATA_DIR}/ppm_updated.csv")
    print("ppm_updated.csv columns:", list(df.columns)); print(df.to_string())
    return df

def pick_ppm_columns(ppm_df):
    cand_match = [c for c in ppm_df.columns if any(k in c.lower() for k in ('cam','phone','device','model'))]
    match_col = cand_match[0] if cand_match else ppm_df.columns[0]
    cand_ppm = [c for c in ppm_df.columns if 'ppm' in c.lower() or 'pixel' in c.lower()]
    ppm_col = cand_ppm[0] if cand_ppm else ppm_df.columns[1]
    print(f"Guessed columns -> camera: '{match_col}', ppm: '{ppm_col}'")
    return match_col, ppm_col

def lookup_ppm(camera_name, ppm_df, match_col, ppm_col):
    norm_name = _canon(camera_name)
    for _, row in ppm_df.iterrows():
        row_name = _canon(row[match_col])
        if row_name and (row_name in norm_name or norm_name in row_name):
            return float(row[ppm_col])
    return float(ppm_df[ppm_col].median())

## 4. Handcrafted texture features (teammate's proven block) + two NEW physical features
(radial power spectrum, Canny-based blob segmentation) as a separately-graded extra block

In [4]:
def fft_dominant_scale_mm(gray_eq, ppm):
    f = np.fft.fft2(gray_eq.astype(np.float32)); fshift = np.fft.fftshift(f)
    power = np.abs(fshift) ** 2
    h, w = gray_eq.shape; cy, cx = h // 2, w // 2
    y, x = np.indices((h, w)); r = np.sqrt((x-cx)**2 + (y-cy)**2).astype(int)
    radial = np.bincount(r.ravel(), power.ravel()) / np.maximum(np.bincount(r.ravel()), 1)
    freqs = np.arange(len(radial)); valid = freqs > 3
    if valid.sum() == 0: return 0.0
    idx = freqs[valid][np.argmax(radial[valid])]
    if idx == 0: return 0.0
    return float((max(h, w) / idx) / ppm)

def extract_texture_features(gray_raw, gray_eq, ppm):
    """Teammate's proven feature set, reused verbatim."""
    feats = {}
    feats['lap_var'] = float(cv2.Laplacian(gray_eq, cv2.CV_64F).var())
    img = gray_eq.astype(np.float32)
    gx = cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=3); gy = cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize=3)
    gmag = np.sqrt(gx**2 + gy**2)
    feats['grad_mean'] = float(gmag.mean()); feats['grad_std'] = float(gmag.std())
    feats['grad_p90'] = float(np.percentile(gmag, 90))
    radius = max(1, int(round(ppm * 0.5)))
    lbp = local_binary_pattern(gray_eq, 8, radius, method='uniform')
    hist, _ = np.histogram(lbp, bins=10, range=(0, 10), density=True)
    for i, h in enumerate(hist): feats[f'lbp_{i}'] = float(h)
    step = max(1, int(round(ppm * 1.0)))
    glcm = graycomatrix(gray_eq, distances=[step], angles=[0, np.pi/2], levels=256, symmetric=True, normed=True)
    feats['glcm_contrast'] = float(graycoprops(glcm, 'contrast').mean())
    feats['glcm_homogeneity'] = float(graycoprops(glcm, 'homogeneity').mean())
    feats['glcm_energy'] = float(graycoprops(glcm, 'energy').mean())
    feats['glcm_correlation'] = float(graycoprops(glcm, 'correlation').mean())
    for mm in [0.2, 0.63, 2.0, 6.3, 20.0]:
        ksize = max(1, int(round(ppm * mm))); ksize = ksize+1 if ksize % 2 == 0 else ksize
        max_k = min(gray_eq.shape); max_k = max_k-1 if max_k % 2 == 0 else max_k
        ksize = max(1, min(ksize, max_k))
        blurred = cv2.GaussianBlur(gray_eq, (ksize, ksize), 0)
        edges = cv2.Canny(blurred, 50, 150)
        feats[f'edge_density_{mm}mm'] = float(edges.mean() / 255.0)
    _, thresh = cv2.threshold(gray_eq, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    contours, _ = cv2.findContours(thresh, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
    areas = [cv2.contourArea(c) / (ppm**2) for c in contours if cv2.contourArea(c) > 0]
    if areas:
        diam = 2 * np.sqrt(np.array(areas) / np.pi)
        feats['blob_diam_mean_mm'] = float(diam.mean()); feats['blob_diam_median_mm'] = float(np.median(diam))
        feats['blob_diam_p90_mm'] = float(np.percentile(diam, 90))
        feats['blob_count_per_mm2'] = float(len(diam) / (gray_eq.shape[0]*gray_eq.shape[1] / ppm**2))
    else:
        feats['blob_diam_mean_mm'] = feats['blob_diam_median_mm'] = 0.0
        feats['blob_diam_p90_mm'] = feats['blob_count_per_mm2'] = 0.0
    feats['fft_dominant_scale_mm'] = fft_dominant_scale_mm(gray_eq, ppm)
    feats['intensity_mean'] = float(gray_raw.mean()); feats['intensity_std'] = float(gray_raw.std())
    return feats

def extract_extra_physical_features(gray_eq, ppm, nb=12):
    """NEW in v5. Two features that measure similar physical quantities to the teammate's
    block, but by a different route -- a second, independently-biased opinion tends to
    combine well with the first when averaged/blended by a regularised model.
      * Radial power spectrum in log-spaced spatial-frequency bands (0.4mm up to the crop's
        own scale) -- a smoother, less threshold-sensitive alternative to the single-peak
        FFT estimate above.
      * Canny-edge + morphological-close blob segmentation (vs. the teammate's Otsu
        threshold) -- a differently-biased estimate of visible particle size."""
    n = min(gray_eq.shape); g = gray_eq[:n, :n].astype(np.float32)
    g = (g - g.mean()) / (g.std() + 1e-6)
    win = np.outer(np.hanning(n), np.hanning(n)).astype(np.float32)
    P = np.abs(np.fft.fft2(g * win)) ** 2
    yx = np.indices((n, n)) - n // 2
    rr = np.sqrt((yx[0]) ** 2 + (yx[1]) ** 2) / n
    edges = np.geomspace(1.0 / (n / 2), 0.5, nb + 1)
    idx = np.clip(np.digitize(rr, edges) - 1, -1, nb - 1)
    m = idx >= 0
    spec = np.log(np.bincount(idx[m], np.fft.fftshift(P)[m], nb) / np.maximum(np.bincount(idx[m], minlength=nb), 1) + 1e-6)
    feats = {f"spec_{i}": float(v) for i, v in enumerate(spec)}
    feats["spec_slope"] = float(np.polyfit(np.log(np.sqrt(edges[:-1]*edges[1:])), spec, 1)[0])

    edges_c = cv2.dilate(cv2.Canny(cv2.GaussianBlur(gray_eq, (0, 0), 1.0), 30, 90), np.ones((3, 3), np.uint8))
    closed = cv2.morphologyEx(edges_c, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8))
    _, _, stats, _ = cv2.connectedComponentsWithStats(255 - cv2.dilate(closed, np.ones((3, 3), np.uint8)))
    areas_px = stats[1:, cv2.CC_STAT_AREA]
    diam_mm = 2 * np.sqrt(np.maximum(areas_px, 1) / np.pi) / ppm
    total_mm2 = gray_eq.shape[0] * gray_eq.shape[1] / ppm ** 2
    for i, (lo, hi) in enumerate(zip([2,4,8,16,32,64], [4,8,16,32,64,128])):
        m2 = (diam_mm >= lo) & (diam_mm < hi)
        feats[f"clast_frac_{lo}_{hi}mm"] = float(areas_px[m2].sum() / ppm**2 / total_mm2) if m2.any() else 0.0
    feats["clast_count_per_mm2"] = float(len(diam_mm) / total_mm2)
    feats["clast_max_mm"] = float(diam_mm.max()) if len(diam_mm) else 0.0
    return feats

def load_and_preprocess(path, target_long_side=1024):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None: return None
    h, w = img.shape[:2]; scale = target_long_side / max(h, w)
    img = cv2.resize(img, (int(w*scale), int(h*scale)), interpolation=cv2.INTER_AREA)
    gray_raw = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray_eq = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gray_raw)
    return gray_raw, gray_eq, scale

def build_handcrafted_feature_df(photo_list, ppm_df, match_col, ppm_col, extra=False):
    rows = []
    for path, sample_id, camera in photo_list:
        result = load_and_preprocess(path)
        if result is None:
            print(f"WARNING: could not read image {os.path.basename(path)}, skipping"); continue
        gray_raw, gray_eq, resize_scale = result
        eff_ppm = lookup_ppm(camera, ppm_df, match_col, ppm_col) * resize_scale
        feats = extract_extra_physical_features(gray_eq, eff_ppm) if extra else extract_texture_features(gray_raw, gray_eq, eff_ppm)
        feats['sample_id'] = sample_id
        rows.append(feats)
    return pd.DataFrame(rows)

## 5. DINOv2 embeddings — kept as an optional, separately-graded feature set
(per the diagnosis: allowed to compete, not assumed to help)

In [5]:
_DINO_MODEL = None
def get_dino_model():
    global _DINO_MODEL
    if _DINO_MODEL is None:
        import torch
        _DINO_MODEL = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14', skip_validation=True, trust_repo=True)
        _DINO_MODEL.eval()
    return _DINO_MODEL

def build_dino_feature_df(paths_and_ids, model):
    import torch, torchvision.transforms as T
    from PIL import Image
    tf = T.Compose([T.Resize(224), T.CenterCrop(224), T.ToTensor(),
                    T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])])
    rows = []
    for path, sample_id in paths_and_ids:
        try:
            x = tf(Image.open(path).convert('RGB')).unsqueeze(0)
            with torch.no_grad():
                emb = model(x).squeeze(0).numpy()
        except Exception as e:
            print(f"WARNING: DINOv2 failed on {os.path.basename(path)}: {e}"); continue
        row = {f'dino_{i}': v for i, v in enumerate(emb)}; row['sample_id'] = sample_id
        rows.append(row)
    return pd.DataFrame(rows)

## 6. Load everything

In [6]:
ppm_df = load_ppm_table()
match_col, ppm_col = pick_ppm_columns(ppm_df)

train_photos = list_photos_recursive(f"{DATA_DIR}/Training-All_Photos_updated", is_train=True)
test_photos = list_photos_recursive(f"{DATA_DIR}/Test_All_Photos", is_train=False)
print(f"\nTrain photos: {len(train_photos)}, test photos: {len(test_photos)}")
assert len(test_photos) > 0, "No test photos found -- check DATA_DIR / folder names before continuing."

print("\nExtracting hand-crafted features (teammate's proven block)...")
train_hc = build_handcrafted_feature_df(train_photos, ppm_df, match_col, ppm_col)
test_hc = build_handcrafted_feature_df(test_photos, ppm_df, match_col, ppm_col)

print("Extracting extra physical features (new in v5: spectrum + Canny blobs)...")
train_extra = build_handcrafted_feature_df(train_photos, ppm_df, match_col, ppm_col, extra=True)
test_extra = build_handcrafted_feature_df(test_photos, ppm_df, match_col, ppm_col, extra=True)

if USE_DINO:
    print("Extracting DINOv2 embeddings (needs internet)...")
    dino_model = get_dino_model()
    train_dino = build_dino_feature_df([(p, sid) for p, sid, _ in train_photos], dino_model)
    test_dino = build_dino_feature_df([(p, sid) for p, sid, _ in test_photos], dino_model)

labels_df = pd.read_csv(f"{DATA_DIR}/Training_labels_updated.csv")
sample_sub = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

def aggregate(df, cols):
    agg_mean = df.groupby('sample_id')[cols].mean().add_suffix('_mean')
    agg_std = df.groupby('sample_id')[cols].std().add_suffix('_std').fillna(0.0)
    return pd.concat([agg_mean, agg_std], axis=1)

hc_cols = [c for c in train_hc.columns if c != 'sample_id']
extra_cols = [c for c in train_extra.columns if c != 'sample_id']
train_hc_agg, test_hc_agg = aggregate(train_hc, hc_cols).reset_index(), aggregate(test_hc, hc_cols).reset_index()
train_extra_agg, test_extra_agg = aggregate(train_extra, extra_cols).reset_index(), aggregate(test_extra, extra_cols).reset_index()

merged_train = train_hc_agg.merge(train_extra_agg, on='sample_id', suffixes=('', '_extra'))
merged_test = test_hc_agg.merge(test_extra_agg, on='sample_id', suffixes=('', '_extra'))
hc_feat_cols = [c for c in train_hc_agg.columns if c != 'sample_id']
extra_feat_cols = [c for c in train_extra_agg.columns if c != 'sample_id']

if USE_DINO:
    dino_cols = [c for c in train_dino.columns if c != 'sample_id']
    train_dino_agg = train_dino.groupby('sample_id')[dino_cols].mean().reset_index()
    test_dino_agg = test_dino.groupby('sample_id')[dino_cols].mean().reset_index()
    merged_train = merged_train.merge(train_dino_agg, on='sample_id', suffixes=('', '_dino'))
    merged_test = merged_test.merge(test_dino_agg, on='sample_id', suffixes=('', '_dino'))
    dino_feat_cols = [c for c in train_dino_agg.columns if c != 'sample_id']

merged_train = merged_train.merge(labels_df, on='sample_id', how='inner')
print(f"\nMerged train samples: {len(merged_train)} (should be 24)")
print(f"Merged test samples: {len(merged_test)} (should be 10)")
assert len(merged_test) == len(sample_sub), "Test samples don't match sample_submission.csv rows."

Y_train = merged_train[SUPPORT_COLS].values.astype(float)
X_test_ids = merged_test['sample_id'].values

feature_sets = {
    'handcrafted': hc_feat_cols,
    'extra_physical': extra_feat_cols,
    'handcrafted+extra': hc_feat_cols + extra_feat_cols,
}
if USE_DINO:
    feature_sets['dino'] = dino_feat_cols
    feature_sets['handcrafted+dino'] = hc_feat_cols + dino_feat_cols
    feature_sets['all'] = hc_feat_cols + extra_feat_cols + dino_feat_cols

ppm_updated.csv columns: ['phone', 'camera', 'width', 'height', 'ppm']
                     phone                   camera  width  height     ppm
0                iPhone 14                iPhone 14   4032    3024  13.942
1                iPhone 16                iPhone 16   5712    4284  19.525
2            Motorola Edge         motorola edge 20   4000    1800  11.492
3  Motorola Edge 60 Fusion  Motorola Edge 60 fusion   4096    2304  12.465
4              Samsung A52                 SM-A525F   9248    6936  26.330
Guessed columns -> camera: 'phone', ppm: 'ppm'

Train photos: 127, test photos: 35

Extracting hand-crafted features (teammate's proven block)...
Extracting extra physical features (new in v5: spectrum + Canny blobs)...
Extracting DINOv2 embeddings (needs internet)...
Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip
Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /root/.

100%|██████████| 84.2M/84.2M [00:00<00:00, 322MB/s]



Merged train samples: 24 (should be 24)
Merged test samples: 10 (should be 10)


## 7. Model zoo — the teammate's 10 regressors, PLUS knn-over-samples (won clearly in
earlier feature spaces), kernel ridge, AND explicit constant mean/median predictors so the
honest CV can show plainly whether anything beats "just guess the average".

In [7]:
class ConstantPredictor:
    """Explicit baseline: always predicts the (mean or median) TRAINING curve, in whichever
    y-space the transform uses. This is exactly what the teammate's script accidentally fell
    back to for every test row (see the bug note at the top) -- included here on purpose and
    on equal footing, so it has to be beaten fairly rather than stumbled into."""
    def __init__(self, kind='mean'): self.kind = kind
    def fit(self, X, Y):
        self.c = (Y.mean(0) if self.kind == 'mean' else np.median(Y, 0)); return self
    def predict(self, X): return np.tile(self.c, (len(X), 1))

class KNNProtoRegressor:
    """k-nearest-neighbour over samples in standardised feature space, blended by inverse
    distance. Always predicts a realistic, in-distribution curve; won clearly against ridge/
    kernel-ridge in the earlier (DINOv2/CLIP-based) feature space explored in v3/v4."""
    def __init__(self, k=3): self.k = k
    def fit(self, X, Y): self.X, self.Y = X, Y; return self
    def predict(self, X):
        d = euclidean_distances(X, self.X); k = min(self.k, d.shape[1])
        out = np.zeros((len(X), self.Y.shape[1]))
        for i in range(len(X)):
            j = np.argsort(d[i])[:k]; w = 1.0 / (d[i, j] + 1e-6); out[i] = (w / w.sum()) @ self.Y[j]
        return out

def build_candidates():
    return {
        'const_mean': lambda: ConstantPredictor('mean'),
        'const_median': lambda: ConstantPredictor('median'),
        'ridge_a15': lambda: Ridge(alpha=15.0),
        'ridge_a30': lambda: Ridge(alpha=30.0),
        'ridge_a50': lambda: Ridge(alpha=50.0),
        'enet_a0.1_l05': lambda: MultiOutputRegressor(ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=20000, tol=1e-3)),
        'enet_a0.15_l04': lambda: MultiOutputRegressor(ElasticNet(alpha=0.15, l1_ratio=0.4, max_iter=20000, tol=1e-3)),
        'pls_2': lambda: PLSRegression(n_components=2),
        'pls_3': lambda: PLSRegression(n_components=3),
        'mtenet_a0.05': lambda: MultiTaskElasticNet(alpha=0.05, l1_ratio=0.5, max_iter=20000, tol=1e-3),
        'mtlasso_a0.05': lambda: MultiTaskLasso(alpha=0.05, max_iter=20000, tol=1e-3),
        'bayesridge': lambda: MultiOutputRegressor(BayesianRidge()),
        'knn_k3': lambda: KNNProtoRegressor(k=3),
        'knn_k5': lambda: KNNProtoRegressor(k=5),
        'krr_a1': lambda: KernelRidge(alpha=1.0, kernel='rbf'),
        'krr_a10': lambda: KernelRidge(alpha=10.0, kernel='rbf'),
    }

## 8. Fit/predict/score plumbing (teammate's structure, reused)

In [8]:
class YScaledModel:
    def __init__(self, base_model_fn, transform):
        self.base_model_fn = base_model_fn; self.transform = transform
    def fit(self, X, Y_raw):
        Yt = self.transform.encode(Y_raw)
        self.y_scaler = StandardScaler(); Yt_s = self.y_scaler.fit_transform(Yt)
        self.model = self.base_model_fn(); self.model.fit(X, Yt_s)
        return self
    def predict(self, X):
        pred_s = np.asarray(self.model.predict(X))
        if pred_s.ndim == 1: pred_s = pred_s.reshape(1, -1)
        return self.transform.decode(self.y_scaler.inverse_transform(pred_s))

def fit_predict_one(X_tr, Y_tr, X_te, model_fn, transform):
    return YScaledModel(model_fn, transform).fit(X_tr, Y_tr).predict(X_te)[0]

def fit_predict_all(X_tr, Y_tr, X_te, model_fn, transform):
    return YScaledModel(model_fn, transform).fit(X_tr, Y_tr).predict(X_te)

def loocv_score_combo(X, Y, model_fn, transform):
    loo = LeaveOneOut(); preds = np.zeros_like(Y)
    for tr, te in loo.split(X):
        preds[te[0]] = fit_predict_one(X[tr], Y[tr], X[te], model_fn, transform)
    return emd_log_score(Y, preds), preds

## 9. Naive comparison across ALL (feature set × model × transform) combinations

In [9]:
candidates = build_candidates()
combos_all = {f"{name}__{t.name}": (fn, t) for name, fn in candidates.items() for t in TRANSFORMS}
print(f"{len(feature_sets)} feature sets x {len(combos_all)} combos = {len(feature_sets)*len(combos_all)} total configurations to score by LOOCV\n")

results = {}
t0 = time.time()
for fs_name, cols in feature_sets.items():
    X_train = merged_train[cols].values
    scaler = StandardScaler(); X_train_s = scaler.fit_transform(X_train)
    scores = {k: loocv_score_combo(X_train_s, Y_train, mf, tf)[0] for k, (mf, tf) in combos_all.items()}
    best_key = min(scores, key=scores.get)
    results[fs_name] = dict(X_train_s=X_train_s, scaler=scaler, scores=scores, best_key=best_key, best_score=scores[best_key])
    top3 = sorted(scores.items(), key=lambda kv: kv[1])[:3]
    print(f"  {fs_name:20s}: best={best_key:22s} LOOCV EMD={scores[best_key]:7.3f}   (top-3: {[(k, round(v,2)) for k,v in top3]})")
print(f"\n({time.time()-t0:.0f}s)")

winner_name = min(results, key=lambda k: results[k]['best_score'])
print(f"\n=> Naive winning feature set: {winner_name} (LOOCV EMD = {results[winner_name]['best_score']:.3f})")
print("Is this actually beating 'just guess the average'? Check the const_mean/const_median rows above for this feature set.")

6 feature sets x 32 combos = 192 total configurations to score by LOOCV

  handcrafted         : best=knn_k5__cumulative     LOOCV EMD= 38.813   (top-3: [('knn_k5__cumulative', np.float64(38.81)), ('knn_k5__delta', np.float64(38.81)), ('knn_k3__cumulative', np.float64(39.54))])
  extra_physical      : best=pls_2__delta           LOOCV EMD= 48.172   (top-3: [('pls_2__delta', np.float64(48.17)), ('pls_2__cumulative', np.float64(48.41)), ('knn_k5__cumulative', np.float64(49.44))])
  handcrafted+extra   : best=pls_2__delta           LOOCV EMD= 40.903   (top-3: [('pls_2__delta', np.float64(40.9)), ('knn_k5__delta', np.float64(41.02)), ('knn_k5__cumulative', np.float64(41.02))])
  dino                : best=knn_k5__cumulative     LOOCV EMD= 41.738   (top-3: [('knn_k5__cumulative', np.float64(41.74)), ('knn_k5__delta', np.float64(41.74)), ('pls_2__cumulative', np.float64(42.07))])
  handcrafted+dino    : best=pls_2__cumulative      LOOCV EMD= 40.438   (top-3: [('pls_2__cumulative', np.float64

## 10. FULL nested cross-validation — re-selects the feature set AND model AND transform
inside every outer fold (stronger than only nesting model+transform on a fixed feature
set, which is what the teammate's script did). This is the number to trust.

In [10]:
def nested_cv_full(feature_sets, Y, candidates, transforms):
    n = len(Y); outer_pred = np.zeros_like(Y); chosen = []
    combos = {f"{name}__{t.name}": (fn, t) for name, fn in candidates.items() for t in transforms}
    for i in range(n):
        mask = np.ones(n, dtype=bool); mask[i] = False
        best_fs, best_key, best_inner = None, None, np.inf
        for fs_name, cols in feature_sets.items():
            X_full = merged_train[cols].values
            scaler = StandardScaler().fit(X_full[mask])
            X_in_s = scaler.transform(X_full[mask])
            for k, (mf, tf) in combos.items():
                s, _ = loocv_score_combo(X_in_s, Y[mask], mf, tf)
                if s < best_inner: best_inner, best_fs, best_key = s, fs_name, k
        mf, tf = combos[best_key]
        X_full = merged_train[feature_sets[best_fs]].values
        scaler = StandardScaler().fit(X_full[mask])
        outer_pred[i] = fit_predict_one(scaler.transform(X_full[mask]), Y[mask], scaler.transform(X_full[i:i+1]), mf, tf)
        chosen.append((best_fs, best_key))
        if i % 6 == 0: print(f"  outer fold {i+1}/{n}...")
    return emd_log_score(Y, outer_pred), chosen

print("Running FULL nested CV (re-selects feature set + model + transform every fold -- this takes a while)...")
t0 = time.time()
nested_score, chosen = nested_cv_full(feature_sets, Y_train, candidates, TRANSFORMS)
print(f"\nNaive best (optimistic):  {results[winner_name]['best_score']:.3f}")
print(f"Nested honest estimate:   {nested_score:.3f}   ({time.time()-t0:.0f}s)")
print(f"Gap: {nested_score - results[winner_name]['best_score']:+.3f}  "
      f"({'small -- trustworthy' if nested_score - results[winner_name]['best_score'] < 5 else 'notable -- treat the naive number with real caution'})")
print(f"Selection stability across folds: {Counter(chosen).most_common(5)}")
const_scores = {k: v for k, v in results[winner_name]['scores'].items() if k.startswith('const_')}
print(f"\nFor reference, plain constant baselines on the naive winner's feature set: {const_scores}")
print("If the nested score above is not clearly better than these, no model here is reliably " 
      "beating 'just guess the average training curve' -- which is what your best submission so far actually was.")

Running FULL nested CV (re-selects feature set + model + transform every fold -- this takes a while)...
  outer fold 1/24...
  outer fold 7/24...
  outer fold 13/24...
  outer fold 19/24...

Naive best (optimistic):  38.813
Nested honest estimate:   48.295   (3537s)
Gap: +9.481  (notable -- treat the naive number with real caution)
Selection stability across folds: [(('handcrafted', 'knn_k5__cumulative'), 8), (('handcrafted', 'knn_k5__delta'), 4), (('handcrafted', 'knn_k3__cumulative'), 3), (('all', 'pls_2__cumulative'), 2), (('handcrafted', 'enet_a0.15_l04__cumulative'), 2)]

For reference, plain constant baselines on the naive winner's feature set: {'const_mean__cumulative': np.float64(89.88886659893313), 'const_mean__delta': np.float64(89.88886659893312), 'const_median__cumulative': np.float64(97.24831519305216), 'const_median__delta': np.float64(95.96063911837415)}
If the nested score above is not clearly better than these, no model here is reliably beating 'just guess the average 

## 11. Optional: blend of the top few DISTINCT configurations
Kept only if it improves on the nested score of the single best model -- not assumed to help.

In [11]:
top_configs = []
seen_fs = set()
all_scored = sorted(
    [(fs, k, v) for fs, r in results.items() for k, v in r['scores'].items()],
    key=lambda t: t[2])
for fs, k, v in all_scored:
    if (fs, k) not in top_configs and len(top_configs) < 5:
        top_configs.append((fs, k))
print("Top-5 distinct (feature_set, model) configs by naive LOOCV:", top_configs)

def loocv_blend_score(configs):
    n = len(Y_train); preds_sum = np.zeros_like(Y_train)
    for fs, key in configs:
        X = merged_train[feature_sets[fs]].values
        scaler = StandardScaler().fit(X)
        Xs = scaler.transform(X)
        mf, tf = combos_all[key]
        _, preds = loocv_score_combo(Xs, Y_train, mf, tf)
        preds_sum += preds
    return emd_log_score(Y_train, preds_sum / len(configs))

blend_score = loocv_blend_score(top_configs)
print(f"Naive LOOCV of blending the top-5 configs: {blend_score:.3f}  (vs single best naive: {results[winner_name]['best_score']:.3f})")
USE_BLEND = blend_score < results[winner_name]['best_score'] - 0.5   # require a real margin, not noise
print(f"-> {'USING the blend' if USE_BLEND else 'using the SINGLE best model'} for the final submission.")

Top-5 distinct (feature_set, model) configs by naive LOOCV: [('handcrafted', 'knn_k5__cumulative'), ('handcrafted', 'knn_k5__delta'), ('handcrafted', 'knn_k3__cumulative'), ('handcrafted', 'knn_k3__delta'), ('all', 'pls_2__cumulative')]
Naive LOOCV of blending the top-5 configs: 37.818  (vs single best naive: 38.813)
-> USING the blend for the final submission.


## 12. Final fit and submission

In [12]:
if USE_BLEND:
    final_pred = np.zeros((len(X_test_ids), 11))
    for fs, key in top_configs:
        X_tr, X_te = merged_train[feature_sets[fs]].values, merged_test[feature_sets[fs]].values
        scaler = StandardScaler().fit(X_tr)
        mf, tf = combos_all[key]
        final_pred += fit_predict_all(scaler.transform(X_tr), Y_train, scaler.transform(X_te), mf, tf)
    final_pred /= len(top_configs)
    print(f"Final submission: blend of {top_configs}")
else:
    fs, key = winner_name, results[winner_name]['best_key']
    X_tr, X_te = merged_train[feature_sets[fs]].values, merged_test[feature_sets[fs]].values
    scaler = StandardScaler().fit(X_tr)
    mf, tf = combos_all[key]
    final_pred = fit_predict_all(scaler.transform(X_tr), Y_train, scaler.transform(X_te), mf, tf)
    print(f"Final submission: single model {fs} / {key}")

final_pred = np.array([enforce_monotonic_and_bounds(r) for r in np.asarray(final_pred).reshape(len(X_test_ids), 11)])
submission = pd.DataFrame(final_pred, columns=SUPPORT_COLS)
submission.insert(0, 'sample_id', X_test_ids)
submission = sample_sub[['sample_id']].merge(submission, on='sample_id', how='left')

missing = submission[SUPPORT_COLS].isna().any(axis=1)
if missing.any():
    print(f"\n!!! {missing.sum()} test rows missing -- this should NOT happen now that the path bug is fixed. "
          f"Investigate before trusting this submission.")
    fallback = enforce_monotonic_and_bounds(Y_train.mean(axis=0))
    for col, val in zip(SUPPORT_COLS, fallback):
        submission.loc[missing, col] = val

assert submission[SUPPORT_COLS].isna().sum().sum() == 0
assert (submission[SUPPORT_COLS].diff(axis=1).iloc[:, 1:] >= -1e-9).all().all()
assert np.allclose(submission['200'], 100.0)
assert missing.sum() == 0, "Test rows are still missing -- the directory-listing fix did not work as expected."

submission.to_csv('submission.csv', index=False)
print(f"\nSaved submission.csv  (naive best-single={results[winner_name]['best_score']:.2f}, nested honest={nested_score:.2f})")
print(submission.round(2).to_string(index=False))

Final submission: blend of [('handcrafted', 'knn_k5__cumulative'), ('handcrafted', 'knn_k5__delta'), ('handcrafted', 'knn_k3__cumulative'), ('handcrafted', 'knn_k3__delta'), ('all', 'pls_2__cumulative')]

Saved submission.csv  (naive best-single=38.81, nested honest=48.29)
               sample_id  0.002  0.0063  0.02  0.063   0.2  0.63     2   6.3    20    63   200
        HPC_Airbus BS6-3   2.55    5.29 12.81  25.06 32.38 38.37 48.32 66.06 87.37 98.54 100.0
          HPC_Audorfring   1.22    2.66  5.96  10.30 14.42 21.65 36.90 62.11 85.15 97.39 100.0
HPC_Muenster_BS6_9_0-10m   1.09    2.38  5.25   9.03 11.94 16.62 29.11 53.56 82.26 96.70 100.0
   HPC_Airbus BS10-4bis7   2.44    5.03 12.21  24.15 32.10 39.62 53.13 73.81 90.64 98.98 100.0
   HPC_Airbus BS12-5bis8   2.86    5.87 14.06  28.02 37.10 43.84 53.34 68.98 87.77 98.65 100.0
 HPC_Kleinkummerfeld 2-2   2.26    4.66 11.37  22.83 33.74 44.76 58.18 77.75 92.08 99.11 100.0
 HPC_Kleinkummerfeld 2-3   1.75    3.67  8.69  16.84 23.96 31